In [ ]:
from azure_blob_user import create_blob_storage_client_via_key, AzureBlobUser
from tqdm import tqdm
from time import sleep
import os

# Identify the container and prefix 
container_name = "mask-blob"
name_prefix = "Batch-"

# USE 
output_zip_dir = r"d:\buffalo_cropped_training"

# Set up the blob_user
blob_storage_client, expiry = create_blob_storage_client_via_key()
blob_user = AzureBlobUser(blob_storage_client, expiry)


def all_files_in_a_folder(target_folder):
    return [f for f in os.listdir(target_folder) if os.path.isfile(os.path.join(target_folder, f))]

def files_to_download_from_blob(blob_user,
                                destination_folder,
                                container_name = container_name,
                                name_prefix = name_prefix,
                                blobs_in_container = None):
    
    # Get all the files we already have 
    already_downloaded_files = all_files_in_a_folder(destination_folder)

    if not blobs_in_container:
        # Get only the blobs we need to download
        blob_user.establish_blob_container(container_name)
        blobs_in_container = blob_user.list_blobs_in_container(name_starts_with=name_prefix, verbose = False, return_blobs=True)
    
    blobs_we_have_not_downloaded = [b.name for b in blobs_in_container if b.name not in already_downloaded_files]
    blobs_we_have_not_downloaded = [b for b in blobs_in_container if b.name not in already_downloaded_files]

    return blobs_we_have_not_downloaded

def try_to_download_blob(blob_user,
                         blob_name,
                         destination_path,
                         number_of_tries = 3,
                         verbose = False,
                         cool_down_after_download = True):
    
    downloaded = False

    if verbose:
        print(f"Attempting download of {blob_name} to {destination_path}")

    for i in range(number_of_tries):

        try:
            blob_user.establish_blob_client(blob_name)
            blob_user.download_blob_client_contents(destination_path)

            if cool_down_after_download:
                blob_user.blob_client.set_standard_blob_tier("Cool")
                
            downloaded = True
            break
        except:
            print(f"Download of {blob_name} failed... trying again. Number of tries is {i+1}")
            sleep(30)

    if downloaded and verbose:
        print(f"Downloaded {blob_name} to {destination_path}!!")    

def download_files_with_restart(blob_user = blob_user,
                                destination_folder = destination_folder,
                                container_name = container_name,
                                name_prefix = name_prefix,
                                blobs_to_download = None,
                                number_of_tries = 3,
                                verbose = False):

    blobs_to_download = files_to_download_from_blob(blob_user, 
                                                    destination_folder,
                                                    container_name = container_name,
                                                    name_prefix = name_prefix,
                                                    blobs_in_container = blobs_to_download)
    
    blob_user.establish_blob_container(container_name)
    destination_path = os.path.join(destination_folder, os.path.basename(blobs_to_download[0].name))

    # Now keep downloading until we run out of things to download
    while blobs_to_download:

        print(f"Starting download of {len(blobs_to_download)} files from blob...")

        for blob in tqdm(blobs_to_download):
            destination_path = os.path.join(destination_folder, os.path.basename(blob.name))

            # Try to download the file a few times before giving up
            try_to_download_blob(blob_user, blob.name, destination_path, number_of_tries, verbose)

        # Check if there are more files to download 
        print(f"Download of {len(blobs_to_download)} files concluded... checking if there are more to download.")
        blobs_to_download = files_to_download_from_blob(blob_user, destination_folder, container_name, name_prefix, blobs_to_download)
        # blobs_to_download = files_to_download_from_blob(blob_user, destination_folder)

In [4]:
from tqdm import tqdm

blob_user.list_containers()
blob_user.establish_blob_container('mask-blob', list_blobs=False)
blobs_in_mask_blob = blob_user.list_blobs_in_container(name_starts_with = 'Batch', verbose = False, return_blobs = True)

Containers:
-azureml
-azureml-blobstore-095ee3eb-cf75-452e-9579-318c12064402
-azureml-blobstore-b12d1401-f7ef-49ad-8b4c-d4622a87b58f
-glint360
-insights-logs-auditevent
-insights-metrics-pt1m
-mask-blob
-nexdata-blob
-temp


In [7]:
blob_user.establish_blob_container('mask-blob', list_blobs=False)
orig_blob = blob_user.list_blobs_in_container(name_starts_with = 'Batch', verbose = False, return_blobs = True)

blob_user.establish_blob_container('nexdata-blob', list_blobs=False)
next_blob = blob_user.list_blobs_in_container(name_starts_with = 'Group', verbose = False, return_blobs = True)

In [13]:
orig_blob_files = ['mask-blob : ' + blob['name'] for blob in orig_blob]
next_blob_files = ['nexdata-blob : ' + blob['name'] for blob in next_blob]

blob_files = orig_blob_files + next_blob_files


with open("blob_files.txt", "w") as f:
    for blob_path in blob_files:
        f.write(f"{blob_path}\n")

In [85]:
download_files_with_restart(blob_user = blob_user,
                            destination_folder = download_folder,
                            container_name = 'mask-blob',
                            name_prefix = "'Batch-'",
                            blobs_to_download = files_to_download,
                            number_of_tries = 3,
                            verbose = False)

Starting download of 9000 files from blob...


 38%|███▊      | 3379/9000 [12:10:15<8:34:20,  5.49s/it]      Unable to stream download: HTTPSConnectionPool(host='ffaimlstorage.blob.core.windows.net', port=443): Read timed out.


Download of Batch-1-m.0glclb_zipped.zip failed... trying again. Number of tries is 1


 43%|████▎     | 3864/9000 [13:21:22<4:37:15,  3.24s/it] Unable to stream download: HTTPSConnectionPool(host='ffaimlstorage.blob.core.windows.net', port=443): Read timed out.


Download of Batch-1-nm4004405_zipped.zip failed... trying again. Number of tries is 1


 94%|█████████▎| 8437/9000 [14:57:28<1:17:59,  8.31s/it] Unable to stream download: HTTPSConnectionPool(host='ffaimlstorage.blob.core.windows.net', port=443): Read timed out.


Download of Batch-10-n000255_zipped.zip failed... trying again. Number of tries is 1


100%|██████████| 9000/9000 [15:03:06<00:00,  6.02s/it]  


Download of 9000 files concluded... checking if there are more to download.


In [45]:
try_to_download_blob(blob_user=blob_user, 
                     blob_name="Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip",
                     destination_path= os.path.join(download_folder, "Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip"),
                     number_of_tries=3,
                     verbose=True)
                     
                     

Attempting download of Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip to D:\datasets\Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip
Downloaded Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip to D:\datasets\Batch-1-0000207-Mustafa-Kemal-Ataturk_zipped.zip!!


In [ ]:
from azure_blob_user import create_blob_storage_client_via_key, AzureBlobUser
from tqdm import tqdm
from time import sleep
import os

# Identify the container and prefix
container_name = os.getenv('CONTAINER_NAME')
name_prefix = os.getenv('PREFIX')
download_folder = os.getenv('DOWNLOAD_FOLDER')

# Set caps for number of tries and download attempts
number_of_tries = int(os.getenv('TRY_LIMIT'))
download_limit = int(os.getenv('DOWNLOAD_LIMIT'))

print("Read parameters!")

# Set where the files are going
# destination_folder = os.path.join(os.getcwd(), "zip_folder")
# download_folder = r"D:\datasets"

# Set up the blob_user
blob_storage_client, expiry = create_blob_storage_client_via_key()
blob_user = AzureBlobUser(blob_storage_client, expiry)

# process files, unzip, etc.
blob_user.establish_blob_container(container_name, list_blobs=False)
print("Identifying Blobs!")
blobs_in_mask_blob = blob_user.list_blobs_in_container(name_starts_with = name_prefix, verbose = False, return_blobs = True)
print(f"Found this many blobs! {len(blobs_in_mask_blob)}")

files_to_download = files_to_download_from_blob(blob_user=blob_user, 
                                                destination_folder=download_folder, 
                                                container_name=container_name, 
                                                name_prefix =  name_prefix,
                                                blobs_in_container=blobs_in_mask_blob)

print(f"I could download this many blobs! {len(files_to_download)}")
print(f"I'm gonna download this many blobs! {len(files_to_download[:download_limit])}")

download_files_with_restart(blob_user = blob_user,
                            destination_folder = download_folder,
                            container_name = container_name,
                            name_prefix = name_prefix,
                            blobs_to_download = files_to_download[:download_limit],
                            number_of_tries = number_of_tries,
                            verbose = False)

Read parameters!
Identifying Blobs!
Found this many blobs! 14998
I could download this many blobs! 14998
I'm gonna download this many blobs! 14998
Starting download of 14998 files from blob...


100%|██████████| 14998/14998 [2:54:08<00:00,  1.44it/s]  


Download of 14998 files concluded... checking if there are more to download.


Read parameters!
Identifying Blobs!
Found this many blobs! 87962
I could download this many blobs! 7922
I'm gonna download this many blobs! 7922
Starting download of 7922 files from blob...


100%|██████████| 7922/7922 [44:30<00:00,  2.97it/s]  


Download of 7922 files concluded... checking if there are more to download.


: 